In [1]:
# Alright lets bring it all together now
# lets define a net
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Lets load our data
import torchvision 
from torchvision.transforms import v2

# torchvision datasets output generally PILIMage images of range [0,1] but then we transform them to tensors
# and normalize them
# why? 
# well two parts
# PIL images are stored with pixel values from 0,255 -> our model doesn't work with just raw rgb or grayscale images - we need to transform them 
# into a tensor so we can conduct our operations as normal
# the second part is that we normalize the pixel values in order to stabilize our training because this results in our 
# gradients being more consistent 
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

  9%|███▋                                   | 16.0M/170M [05:54<56:59, 45.2kB/s]


KeyboardInterrupt: 

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        self.conv1 = nn.conv2d(3,6,5) # remember rgb input channels, feature map, kernel size
        self.pool = nn.MaxPool2d(2) # same thing as F.max_pool2d -> just reusable
        self.conv2= nn.conv2d(6,16,5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    

In [ ]:
# define our loss - in this example we will use cross entropy and our optimizer
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=.001, momentum=.9)


In [ ]:
# Now we train the network
for epoch in range(4):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #print some stats
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

In [ ]:
# Now lets save our model
PATH = './cifar_net.pt'
torch.save(net.state_dict(), PATH)

In [ ]:
# and then test the net on the test data
images, labels = next(iter(testloader))
# print images
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

In [ ]:
# now lets reload the model
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))
# and sick it on the test data
_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(4)))

In [ ]:
# to get a full picture of how our model performs on the test data we now loop through it
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, label = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += label.size(0)
        correct += (predicted == labels).sum().item() # df of where predicted == labels sum the number of rows and fetch the number
        
print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %') 